In [12]:
import pandas as pd
import numpy as np

!wget -O heart.csv "https://raw.githubusercontent.com/mclarenh/Machine-Learning-Final-project/refs/heads/main/data/heart.csv"

df = pd.read_csv("heart.csv")

print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

--2026-11-05 00:31:45--  https://raw.githubusercontent.com/mclarenh/Machine-Learning-Final-project/refs/heads/main/data/heart.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35921 (35K) [text/plain]
Saving to: ‘heart.csv’

heart.csv           100%[===================>]  35.08K  --.-KB/s    in 0s      

2026-11-05 00:31:45 (90.2 MB/s) - ‘heart.csv’ saved [35921/35921]

Dataset dimensions: (918, 12)

Column names:
['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)

print("\nDevelopment class proportions:")
print(y_dev.value_counts(normalize=True))

Development set: (734, 11)
Test set: (184, 11)

Development class proportions:
HeartDisease
1    0.553134
0    0.446866
Name: proportion, dtype: float64


In [14]:
median_cholesterol = X_dev.loc[
    X_dev["Cholesterol"] != 0, "Cholesterol"
].median()

print("Median cholesterol:", median_cholesterol)

X_dev = X_dev.copy()
X_test = X_test.copy()

X_dev["Cholesterol"] = X_dev["Cholesterol"].replace(
    0, median_cholesterol
)

X_test["Cholesterol"] = X_test["Cholesterol"].replace(
    0, median_cholesterol
)

print("Remaining zeros in development:",
      (X_dev["Cholesterol"] == 0).sum())

print("Remaining zeros in test:",
      (X_test["Cholesterol"] == 0).sum())

Median cholesterol: 238.0
Remaining zeros in development: 0
Remaining zeros in test: 0


In [15]:
categorical_cols = [
    "Sex",
    "ChestPainType",
    "FastingBS",
    "RestingECG",
    "ExerciseAngina",
    "ST_Slope"
]

X_dev = pd.get_dummies(
    X_dev,
    columns=categorical_cols,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    dtype=int
)

X_test = X_test.reindex(
    columns=X_dev.columns,
    fill_value=0
)

print("Development shape:", X_dev.shape)
print("Test shape:", X_test.shape)

display(X_dev.head())

Development shape: (734, 21)
Test shape: (184, 21)


,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,...,FastingBS_0,FastingBS_1,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
485,63,139,217,128,1.2,0,1,0,1,0,...,0,1,0,0,1,0,1,0,1,0
486,55,110,214,180,0.4,0,1,0,1,0,...,0,1,0,0,1,1,0,0,0,1
117,59,130,338,130,1.5,1,0,1,0,0,...,0,1,0,0,1,0,1,0,1,0
361,47,160,238,124,0.0,0,1,1,0,0,...,1,0,0,1,0,0,1,0,1,0
296,50,145,238,139,0.7,0,1,1,0,0,...,0,1,0,1,0,0,1,0,1,0


In [16]:
from sklearn.model_selection import StratifiedKFold

# Create 5 stratified folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Generate folds using development data only
folds = list(cv.split(X_dev, y_dev))

# Inspect the folds
for i, (train_idx, val_idx) in enumerate(folds):
    print(f"Fold {i+1}:")
    print("  Training rows:", len(train_idx))
    print("  Validation rows:", len(val_idx))
    print()

Fold 1:
  Training rows: 587
  Validation rows: 147

Fold 2:
  Training rows: 587
  Validation rows: 147

Fold 3:
  Training rows: 587
  Validation rows: 147

Fold 4:
  Training rows: 587
  Validation rows: 147

Fold 5:
  Training rows: 588
  Validation rows: 146



In [17]:
from sklearn.metrics import accuracy_score

baseline_accuracies = []

for train_idx, val_idx in folds:

    y_train = y_dev.iloc[train_idx]
    y_val = y_dev.iloc[val_idx]

    # Always predict the most common training class
    majority_class = y_train.mode()[0]

    predictions = [majority_class] * len(y_val)

    baseline_accuracies.append(
        accuracy_score(y_val, predictions)
    )

print("Fold accuracies:", baseline_accuracies)
print("Average baseline accuracy:", np.mean(baseline_accuracies))

Fold accuracies: [0.5578231292517006, 0.5510204081632653, 0.5510204081632653, 0.5510204081632653, 0.5547945205479452]
Average baseline accuracy: 0.5531357748578885


In [18]:
import os

os.makedirs("data/processed", exist_ok=True)

X_dev.to_csv("data/processed/X_dev.csv", index=False)
y_dev.to_csv("data/processed/y_dev.csv", index=False)

X_test.to_csv("data/processed/X_test.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

In [19]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Use A's objects from the setup cells above
X = X_dev.to_numpy(dtype=float)
y = y_dev.to_numpy(dtype=int)

print(X.shape, y.shape)   # should be (734, 21) and (734,)
print([len(v) for t, v in folds])   # should be [147, 147, 147, 147, 146]

(734, 21) (734,)
[147, 147, 147, 147, 146]


In [20]:
# k-NN from HW3
def majority(labels):
    return np.bincount(labels).argmax()

def knn_predict(X_train, y_train, X_query, k):
    predictions = []
    for x in X_query:
        distances = np.sqrt(np.sum((X_train - x) ** 2, axis=1))   # distance to every training row
        nearest = np.argsort(distances)[:k]                       # k closest rows
        predictions.append(majority(y_train[nearest]))            # majority label
    return np.array(predictions)

In [21]:
def cv_knn(X, y, k):
    scores = []
    for train_rows, val_rows in folds:
        X_tr, y_tr = X[train_rows], y[train_rows]
        X_va, y_va = X[val_rows], y[val_rows]
        mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)
        std[std == 0] = 1
        X_tr = (X_tr - mean) / std
        X_va = (X_va - mean) / std
        y_pred = knn_predict(X_tr, y_tr, X_va, k)
        scores.append(np.mean(y_pred == y_va))
    return np.mean(scores), np.std(scores)

cv_mean, cv_sd = cv_knn(X, y, 5)
print('k-NN (k = 5): CV accuracy', round(cv_mean, 4), ' s.d.', round(cv_sd, 4))

k-NN (k = 5): CV accuracy 0.8379  s.d. 0.034


In [22]:
def cv_logreg(X, y):
    scores = []
    for train_rows, val_rows in folds:
        X_tr, y_tr = X[train_rows], y[train_rows]
        X_va, y_va = X[val_rows], y[val_rows]
        mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)
        std[std == 0] = 1
        X_tr = (X_tr - mean) / std
        X_va = (X_va - mean) / std
        model = LogisticRegression(max_iter=1000)
        model.fit(X_tr, y_tr)
        scores.append(np.mean(model.predict(X_va) == y_va))
    return np.mean(scores), np.std(scores)

cv_mean, cv_sd = cv_logreg(X, y)
print('Logistic regression: CV accuracy', round(cv_mean, 4), ' s.d.', round(cv_sd, 4))

Logistic regression: CV accuracy 0.8474  s.d. 0.0299
